# Strict HMC vs Score-Compressed Likelihood

This notebook compares the improved strict HMC benchmark against the score compression + Fisher parameter whitening + simulation-calibrated Gaussian/Student-t compressed likelihood for the analytical Cl datavector. The compressed-likelihood rows report the total number of simulations used to calibrate the compressed-summary covariance.


## Key Run Summary

Strict HMC uses no simulations; it is a likelihood benchmark. Its diagnostics are:

- divergences: `0`
- max Rhat: `1.0`
- min ESS bulk: `2468.0`

Compressed-likelihood comparison against strict HMC:

| method | total simulations | likelihood | theta shift / sigma | nu shift / sigma | theta width | nu width | q width | d width |
|---|---:|---|---:|---:|---:|---:|---:|---:|
| Student-t 128 sims | 128 | student_t | -0.029 | -0.030 | 1.078 | 1.086 | 0.985 | 1.078 |
| Gaussian 128 sims | 128 | gaussian | -0.024 | -0.025 | 1.070 | 1.077 | 0.969 | 1.070 |
| Student-t 256 sims | 256 | student_t | -0.010 | -0.011 | 1.031 | 1.041 | 1.015 | 1.031 |
| Gaussian 256 sims | 256 | gaussian | -0.008 | -0.008 | 1.027 | 1.036 | 1.008 | 1.027 |


## Posterior Triangle

The exact score posterior is included as a zero-simulation reference for the linearized analytical problem.

![Posterior triangle](outputs/theory_sbi/strict_hmc_compressed_comparison/figures/posterior_triangle_hmc_compressed.png)


## Constrained Direction

The narrow eigen-direction is where SBI calibration problems showed up most clearly. The 256-simulation compressed likelihood agrees with strict HMC at the percent level in this direction.

![Constrained direction](outputs/theory_sbi/strict_hmc_compressed_comparison/figures/constrained_direction_hmc_compressed.png)


## Width Ratios

The 128-simulation compressed likelihood is already close, but 256 simulations is the safer default for map-SBI because the broad-direction width error is smaller.

![Width ratios](outputs/theory_sbi/strict_hmc_compressed_comparison/figures/width_ratios_hmc_compressed.png)


## Reproduce The Plots

Run the cells below from this notebook directory to regenerate the table and figures.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from getdist import plots, MCSamples

NOTEBOOK_DIR = Path.cwd()
BASE = NOTEBOOK_DIR / "outputs" / "theory_sbi"
if not BASE.exists():
    BASE = Path("/mnt/ceph/users/spandey/ltu-godmax/GODMAX/notebooks/SBI_validate/outputs/theory_sbi")

HMC_RUN = "joint_gg_gy_gtau_gkappa_linearized_hmc_strict"
COMPRESSED_RUNS = {
    "Student-t 128 sims": "compressed_likelihood_student_t_nsim128",
    "Gaussian 128 sims": "compressed_likelihood_gaussian_nsim128",
    "Student-t 256 sims": "compressed_likelihood_student_t_nsim256",
    "Gaussian 256 sims": "compressed_likelihood_gaussian_nsim256",
}
FIDUCIAL = np.array([2.0, -0.1])


In [ ]:
def load_hmc(run=HMC_RUN):
    z = np.load(BASE / run / "hmc_samples.npz")
    return np.column_stack([z["samples_theta_ej_0"], z["samples_nu_theta_ej_M"]])

def load_compressed(run, key="samples"):
    return np.load(BASE / run / "compressed_likelihood_samples.npz", allow_pickle=True)[key]

def load_diag(run):
    return json.loads((BASE / run / "compressed_likelihood_diagnostics.json").read_text())

hmc = load_hmc()
hmc_diag = json.loads((BASE / HMC_RUN / "hmc_diagnostics.json").read_text())
hmc_npz = np.load(BASE / HMC_RUN / "hmc_samples.npz")
print("strict HMC divergences:", int(hmc_npz["extra_diverging"].sum()))
print("strict HMC max Rhat:", hmc_diag.get("max_rhat"))
print("strict HMC min ESS bulk:", hmc_diag.get("min_ess_bulk"))
print("strict HMC mean/std:", hmc.mean(axis=0), hmc.std(axis=0))


In [ ]:
ref_cov = np.cov(hmc.T)
evals, evecs = np.linalg.eigh(ref_cov)
constrained_vec = evecs[:, 0]
degenerate_vec = evecs[:, 1]
if constrained_vec[1] > 0:
    constrained_vec = -constrained_vec
if degenerate_vec[0] < 0:
    degenerate_vec = -degenerate_vec
qh = (hmc - FIDUCIAL[None, :]) @ constrained_vec
dh = (hmc - FIDUCIAL[None, :]) @ degenerate_vec

rows = []
for label, run in COMPRESSED_RUNS.items():
    samples = load_compressed(run)
    diag = load_diag(run)
    q = (samples - FIDUCIAL[None, :]) @ constrained_vec
    d = (samples - FIDUCIAL[None, :]) @ degenerate_vec
    rows.append({
        "method": label,
        "total simulations": int(diag["nsim"]),
        "likelihood": diag["likelihood"],
        "theta shift / sigma": (samples[:, 0].mean() - hmc[:, 0].mean()) / hmc[:, 0].std(),
        "nu shift / sigma": (samples[:, 1].mean() - hmc[:, 1].mean()) / hmc[:, 1].std(),
        "theta width ratio": samples[:, 0].std() / hmc[:, 0].std(),
        "nu width ratio": samples[:, 1].std() / hmc[:, 1].std(),
        "q width ratio": q.std() / qh.std(),
        "d width ratio": d.std() / dh.std(),
    })
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(row)


In [ ]:
names = ["theta_ej_0", "nu_theta_ej_M"]
labels = [r"\theta_{\rm ej,0}", r"\nu^M_{\theta_{\rm ej}}"]
plot_specs = [
    ("Strict HMC", hmc, "#1f77b4", False),
    ("Exact score posterior", load_compressed("compressed_likelihood_student_t_nsim256", key="samples_exact"), "black", False),
    ("Student-t 256 sims", load_compressed("compressed_likelihood_student_t_nsim256"), "#d62728", True),
    ("Gaussian 256 sims", load_compressed("compressed_likelihood_gaussian_nsim256"), "#2ca02c", False),
]
mc = [MCSamples(samples=s, names=names, labels=labels, label=label, settings={"ignore_rows": 0.0}) for label, s, _, _ in plot_specs]
g = plots.get_subplot_plotter(width_inch=7.0)
g.settings.figure_legend_frame = False
g.settings.alpha_filled_add = 0.20
g.triangle_plot(
    mc,
    filled=[filled for _, _, _, filled in plot_specs],
    contour_colors=[color for _, _, color, _ in plot_specs],
    legend_labels=[label for label, _, _, _ in plot_specs],
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0), constrained_layout=True)
bins = np.linspace(np.percentile(qh, 0.3), np.percentile(qh, 99.7), 65)
axes[0].hist(qh, bins=bins, density=True, histtype="step", lw=2.0, color="#1f77b4", label="Strict HMC")
for label, run, color in [
    ("Student-t 256 sims", "compressed_likelihood_student_t_nsim256", "#d62728"),
    ("Gaussian 256 sims", "compressed_likelihood_gaussian_nsim256", "#2ca02c"),
    ("Student-t 128 sims", "compressed_likelihood_student_t_nsim128", "#ff7f0e"),
]:
    samples = load_compressed(run)
    q = (samples - FIDUCIAL[None, :]) @ constrained_vec
    axes[0].hist(q, bins=bins, density=True, histtype="step", lw=1.5, label=label, color=color)
axes[0].set_xlabel(r"$q = (\theta-\theta_{\rm fid})\cdot e_{\rm constrained}$")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)
axes[0].set_title("Most constrained direction")

rng = np.random.default_rng(12)
for label, samples, color, alpha in [
    ("Strict HMC", hmc, "#1f77b4", 0.18),
    ("Student-t 256 sims", load_compressed("compressed_likelihood_student_t_nsim256"), "#d62728", 0.12),
    ("Gaussian 256 sims", load_compressed("compressed_likelihood_gaussian_nsim256"), "#2ca02c", 0.12),
]:
    q = (samples - FIDUCIAL[None, :]) @ constrained_vec
    keep = rng.choice(len(samples), size=min(5000, len(samples)), replace=False)
    axes[1].scatter(samples[keep, 0], q[keep], s=3, alpha=alpha, label=label, color=color, rasterized=True)
axes[1].axhline(0.0, color="black", ls="--", lw=0.8)
axes[1].axvline(FIDUCIAL[0], color="black", ls="--", lw=0.8)
axes[1].set_xlabel(r"$\theta_{\rm ej,0}$")
axes[1].set_ylabel(r"$q$ constrained")
axes[1].legend(fontsize=8)
axes[1].set_title("Thin direction in physical coordinates")
plt.show()
